In [ ]:
# In ra thông tin ban đầu về môi trường, giúp xác nhận notebook đã được thiết lập đúng.
print("Everything looks good to go!")

In [ ]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import os


def find_project_root():
    """
    Tìm project root bằng cách đi ngược từ thư mục hiện tại
    và tìm các marker đặc trưng của project.
    """
    markers = ["src", "configs", "notebooks", "data", "outputs"]
    cwd = Path.cwd().resolve()

    for candidate in [cwd] + list(cwd.parents):
        score = sum((candidate / marker).exists() for marker in markers)
        if score >= 3:
            return candidate

    raise FileNotFoundError(
        f"Không tìm thấy project root từ cwd={cwd}. "
        "Hãy mở notebook từ bên trong ticket_suspicion_project."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ PROJECT_ROOT:", PROJECT_ROOT)
print("✅ Current working directory:", Path.cwd())
print("✅ Python executable:", sys.executable)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
import traceback
from datetime import datetime

CHECK_RESULTS = []


def record_check(name, passed, message="", detail=None):
    CHECK_RESULTS.append({
        "check_name": name,
        "passed": bool(passed),
        "message": message,
        "detail": detail
    })

    icon = "✅ PASS" if passed else "❌ FAIL"
    print(f"{icon} | {name}")

    if message:
        print("   ", message)

    if detail is not None:
        print("   ", detail)


def safe_check(name, func):
    try:
        result = func()

        if isinstance(result, tuple):
            passed, message, detail = result
        elif isinstance(result, bool):
            passed, message, detail = result, "", None
        else:
            passed, message, detail = True, str(result), None

        record_check(name, passed, message, detail)

    except Exception as e:
        record_check(
            name,
            False,
            message=str(e),
            detail=traceback.format_exc()
        )


print("✅ Check framework ready.")
print("Started at:", datetime.now())


In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
def check_project_structure():
    required_dirs = [
        "notebooks",
        "src",
        "configs",
        "data/raw",
        "data/processed",
        "data/simulation",
        "outputs",
        "outputs/frames",
        "outputs/tracks",
        "outputs/tracklets",
        "outputs/crops",
        "outputs/embeddings",
        "outputs/associations",
        "outputs/global_ids",
        "outputs/events",
        "outputs/journeys",
        "outputs/alerts",
        "outputs/evidence",
        "outputs/visualizations",
    ]

    missing = []

    for d in required_dirs:
        path = PROJECT_ROOT / d
        if not path.exists():
            missing.append(d)

    if missing:
        return False, "Thiếu thư mục.", missing

    return True, "Cấu trúc thư mục cơ bản đầy đủ.", None


safe_check("Project folder structure", check_project_structure)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_src_files():
    required_files = [
        "src/config.py",
        "src/io_utils.py",
        "src/video_sync.py",
        "src/roi_utils.py",
        "src/journey_state.py",
        "src/scoring.py",
        "src/evidence.py",
        "src/dataset_preflight.py",
    ]

    missing = []

    for f in required_files:
        if not (PROJECT_ROOT / f).exists():
            missing.append(f)

    if missing:
        return False, "Thiếu file trong src/.", missing

    return True, "Các file src cần thiết đều tồn tại.", None


safe_check("Source files", check_src_files)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_config_files():
    required_configs = [
        "configs/pipeline_config.yaml",
        "configs/camera_config.yaml",
        "configs/roi_config.yaml",
        "configs/scoring_config.yaml",
    ]

    missing = []

    for f in required_configs:
        if not (PROJECT_ROOT / f).exists():
            missing.append(f)

    if missing:
        return False, "Thiếu config YAML.", missing

    return True, "Đầy đủ config YAML.", None


safe_check("Config files", check_config_files)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_internal_imports():
    from src.config import load_yaml
    from src.io_utils import save_csv, load_csv
    from src.video_sync import find_img1_dirs, read_mot_seqinfo
    from src.dataset_preflight import scan_mot_dataset, print_preflight_summary, save_preflight_report

    return True, "Import module nội bộ thành công.", None


safe_check("Internal Python imports", check_internal_imports)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_minimal_python_packages():
    packages = {
        "pandas": "pandas",
        "numpy": "numpy",
        "yaml": "pyyaml",
        "PIL": "pillow",
    }

    missing = []

    for import_name, pip_name in packages.items():
        try:
            __import__(import_name)
        except Exception as e:
            missing.append({
                "import": import_name,
                "pip_package": pip_name,
                "error": str(e)
            })

    if missing:
        return False, "Thiếu thư viện tối thiểu để check dataset.", missing

    return True, "Thư viện tối thiểu để check dataset đã đủ.", None


safe_check("Minimal Python packages", check_minimal_python_packages)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_python_packages():
    packages = {
        "pandas": "pandas",
        "numpy": "numpy",
        "yaml": "pyyaml",
        "PIL": "pillow",
        "cv2": "opencv-python",
        "matplotlib": "matplotlib",
        "sklearn": "scikit-learn",
        "torch": "torch",
        "ultralytics": "ultralytics",
    }

    missing = []

    for import_name, pip_name in packages.items():
        try:
            __import__(import_name)
        except Exception as e:
            missing.append({
                "import": import_name,
                "pip_package": pip_name,
                "error": str(e)
            })

    if missing:
        return False, "Thiếu một số thư viện Python.", missing

    return True, "Các thư viện Python chính đều import được.", None


safe_check("Python packages", check_python_packages)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
import requests


DATASET_URLS = {
    "MOT17 official": "https://motchallenge.net/data/MOT17/",
    "MOT20 official": "https://motchallenge.net/data/MOT20/",
    "WILDTRACK official": "https://www.epfl.ch/labs/cvlab/data/data-wildtrack/",
    "Kaggle MOT17 notebook": "https://www.kaggle.com/code/stpeteishii/motchallenge-mot17-data-view",
}


def check_url_exists(url, timeout=15):
    try:
        response = requests.head(url, allow_redirects=True, timeout=timeout)

        # Một số site chặn HEAD, thử GET nhẹ
        if response.status_code in [403, 405, 429] or response.status_code >= 500:
            response = requests.get(url, allow_redirects=True, timeout=timeout, stream=True)

        return {
            "url": url,
            "status_code": response.status_code,
            "ok": response.status_code < 400,
            "final_url": response.url
        }

    except Exception as e:
        return {
            "url": url,
            "status_code": None,
            "ok": False,
            "error": str(e),
            "final_url": None
        }


def check_dataset_urls():
    results = {}

    for name, url in DATASET_URLS.items():
        results[name] = check_url_exists(url)

    # Không bắt buộc tất cả URL đều pass.
    # Chỉ cần ít nhất Kaggle hoặc MOT official sống là được.
    any_ok = any(v["ok"] for v in results.values())

    if not any_ok:
        return False, "Không URL dataset nào truy cập được.", results

    failed = {k: v for k, v in results.items() if not v["ok"]}

    if failed:
        return True, "Một số URL không truy cập được, nhưng vẫn có URL còn sống.", results

    return True, "Tất cả URL dataset đều truy cập được.", results


safe_check("Dataset URLs", check_dataset_urls)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_mot17_local_dataset():
    from src.dataset_preflight import (
        scan_mot_dataset,
        save_preflight_report
    )

    mot_root = PROJECT_ROOT / "data/raw/MOT17"

    if not mot_root.exists():
        return False, f"Không tìm thấy MOT17 tại: {mot_root}", None

    report_df = scan_mot_dataset(mot_root)

    out_path = PROJECT_ROOT / "outputs/dataset_preflight_report.csv"
    save_preflight_report(report_df, out_path)

    if "status" not in report_df.columns:
        return False, "Report không có cột status.", report_df.head().to_dict("records")

    ok_count = int((report_df["status"] == "OK").sum())
    total = len(report_df)

    if ok_count == 0:
        return False, "Không có sequence MOT17 nào OK.", report_df.to_dict("records")

    summary = {
        "mot_root": str(mot_root),
        "total_sequences_found": total,
        "ok_sequences": ok_count,
        "report_saved_to": str(out_path),
        "status_counts": report_df["status"].value_counts().to_dict()
    }

    return True, f"Tìm thấy {ok_count}/{total} sequence MOT17 OK.", summary


safe_check("Local MOT17 dataset", check_mot17_local_dataset)


In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
import pandas as pd

report_path = PROJECT_ROOT / "outputs/dataset_preflight_report.csv"

if report_path.exists():
    report_df = pd.read_csv(report_path)

    display_cols = [
        "sequence_name",
        "num_images",
        "frameRate",
        "seqLength",
        "imWidth",
        "imHeight",
        "has_seqinfo",
        "has_gt_txt",
        "sample_image_readable",
        "status",
        "img1_dir",
    ]

    display_cols = [c for c in display_cols if c in report_df.columns]

    print("Dataset preflight report:")
    display(report_df[display_cols])
else:
    print("❌ Chưa có dataset_preflight_report.csv")


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_camera_config_matches_dataset():
    import yaml
    import pandas as pd

    camera_config_path = PROJECT_ROOT / "configs/camera_config.yaml"
    report_path = PROJECT_ROOT / "outputs/dataset_preflight_report.csv"

    if not camera_config_path.exists():
        return False, "Không tìm thấy camera_config.yaml.", None

    if not report_path.exists():
        return False, "Không tìm thấy dataset_preflight_report.csv. Hãy chạy check MOT17 trước.", None

    with open(camera_config_path, "r", encoding="utf-8") as f:
        camera_cfg = yaml.safe_load(f)

    report_df = pd.read_csv(report_path)

    cam_cfg = camera_cfg["cameras"].get("cam_entrance_01", {})
    sequence_keyword = cam_cfg.get("sequence_keyword", None)

    if not sequence_keyword:
        return True, "Không đặt sequence_keyword, pipeline sẽ chọn sequence đầu tiên.", None

    matched = report_df[
        report_df["sequence_name"].astype(str).str.contains(sequence_keyword, na=False)
    ]

    if len(matched) == 0:
        available = report_df["sequence_name"].dropna().tolist()
        return False, f"Không tìm thấy sequence_keyword={sequence_keyword}", available

    ok_matched = matched[matched["status"] == "OK"]

    if len(ok_matched) == 0:
        return False, f"Có sequence {sequence_keyword} nhưng status không OK.", matched.to_dict("records")

    return True, f"sequence_keyword={sequence_keyword} tồn tại và OK.", ok_matched[["sequence_name", "num_images", "status"]].to_dict("records")


safe_check("camera_config sequence_keyword", check_camera_config_matches_dataset)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_sample_frames_readable():
    import pandas as pd
    from PIL import Image

    report_path = PROJECT_ROOT / "outputs/dataset_preflight_report.csv"

    if not report_path.exists():
        return False, "Không có dataset_preflight_report.csv.", None

    report_df = pd.read_csv(report_path)
    ok_df = report_df[report_df["status"] == "OK"].copy()

    if len(ok_df) == 0:
        return False, "Không có sequence OK để đọc thử.", None

    img1_dir = Path(ok_df.iloc[0]["img1_dir"])

    images = sorted(
        list(img1_dir.glob("*.jpg")) +
        list(img1_dir.glob("*.jpeg")) +
        list(img1_dir.glob("*.png"))
    )

    if len(images) == 0:
        return False, "Không có ảnh trong img1_dir.", str(img1_dir)

    checked = []

    for img_path in images[:5]:
        with Image.open(img_path) as img:
            checked.append({
                "file": img_path.name,
                "size": img.size
            })

    return True, "Đọc thử 5 ảnh đầu tiên thành công.", checked


safe_check("Sample frame readability", check_sample_frames_readable)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_required_notebooks():
    required_notebooks = [
        "01_setup_and_dataset.ipynb",
        "02_input_video_and_sync.ipynb",
        "03_detection_tracking.ipynb",
        "04_tracklet_crop_extraction.ipynb",
        "05_reid_embedding.ipynb",
        "06_multicamera_association_global_id.ipynb",
        "07_roi_event_journey_scoring.ipynb",
        "08_alert_visual_evidence_human_review.ipynb",
        "09_evaluation_visualization.ipynb",
    ]

    missing = []

    for nb in required_notebooks:
        if not (PROJECT_ROOT / "notebooks" / nb).exists():
            missing.append(nb)

    if missing:
        return False, "Thiếu notebook.", missing

    return True, "Đủ 9 notebook pipeline.", None


safe_check("Pipeline notebooks", check_required_notebooks)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_output_write_permission():
    test_file = PROJECT_ROOT / "outputs" / "_write_test.txt"

    try:
        test_file.parent.mkdir(parents=True, exist_ok=True)
        test_file.write_text("write test ok", encoding="utf-8")

        content = test_file.read_text(encoding="utf-8")
        test_file.unlink()

        if content != "write test ok":
            return False, "Ghi được nhưng đọc lại sai nội dung.", None

        return True, "Có quyền ghi vào outputs/.", None

    except Exception as e:
        return False, "Không thể ghi vào outputs/.", str(e)


safe_check("Output write permission", check_output_write_permission)


In [ ]:
# Xác nhận môi trường, cấu hình và dữ liệu ban đầu trước khi chạy toàn bộ pipeline.
def check_yolo_init():
    try:
        from ultralytics import YOLO

        model = YOLO("yolov8n.pt")

        return True, "YOLOv8n khởi tạo thành công.", str(type(model))

    except Exception as e:
        return False, "Không khởi tạo được YOLOv8n.", str(e)


safe_check("YOLO model initialization", check_yolo_init)


In [ ]:
def check_camera_config_matches_dataset():
    import yaml

    report_path = PROJECT_ROOT / "outputs/dataset_preflight_report.csv"
    if not report_path.exists():
        return False, "Không có dataset_preflight_report.csv.", None

    report_df = pd.read_csv(report_path)

    try:
        cam_cfg = yaml.safe_load((PROJECT_ROOT / "configs" / "camera_config.yaml").read_text()) or {}
    except Exception:
        cam_cfg = {}

    sequence_keyword = cam_cfg.get("sequence_keyword", None)

    if not sequence_keyword:
        available = report_df["sequence_name"].dropna().unique().tolist()
        return True, "Không đặt sequence_keyword — pipeline sẽ xử lý tất cả sequence tìm thấy.", available

    matched = report_df[report_df["sequence_name"].astype(str).str.contains(sequence_keyword, na=False)]

    if len(matched) == 0:
        available = report_df["sequence_name"].dropna().tolist()
        return False, f"Không tìm thấy sequence_keyword={sequence_keyword}", available

    ok_matched = matched[matched["status"] == "OK"]

    if len(ok_matched) == 0:
        return False, f"Có sequence {sequence_keyword} nhưng status không OK.", matched.to_dict("records")

    return True, f"sequence_keyword={sequence_keyword} tồn tại và OK.", ok_matched[["sequence_name", "num_images", "status"]].to_dict("records")


In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
import pandas as pd

results_df = pd.DataFrame(CHECK_RESULTS)

out_path = PROJECT_ROOT / "outputs" / "00_preflight_check_results.csv"
results_df.to_csv(out_path, index=False)

print("Saved preflight results to:", out_path)

display(results_df)

num_pass = int(results_df["passed"].sum())
num_total = len(results_df)
num_fail = num_total - num_pass

print("=" * 80)
print("PREFLIGHT FINAL SUMMARY")
print("=" * 80)
print(f"PASS: {num_pass}/{num_total}")
print(f"FAIL: {num_fail}/{num_total}")

if num_fail == 0:
    print("✅ READY TO RUN PIPELINE")
    print("Bạn có thể chạy tiếp 01 → 09.")
else:
    print("❌ NOT READY")
    print("Cần xử lý các check FAIL trước khi chạy pipeline chính.")

    failed_df = results_df[results_df["passed"] == False]
    display(failed_df[["check_name", "message", "detail"]])


In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
failed = results_df[results_df["passed"] == False]

if len(failed) > 0:
    raise RuntimeError(
        f"Preflight failed: {len(failed)} check(s) failed. "
        "Hãy xem bảng lỗi phía trên trước khi chạy 01-09."
    )

print("✅ All checks passed. Safe to continue.")
